In [ ]:
#Distribution Shift Analysis

In [1]:
"""
Distribution Shift Analysis — fNIRS HbO/HbR
============================================
برای هر subject در هر session، میانگین و واریانس HbO و HbR
را در هر کلاس محاسبه می‌کند و با آزمون آماری session 3
را با sessions 1 و 2 مقایسه می‌کند.

ساختار فایل‌های ورودی (از preprocessing):
  {OUTPUT_PATH}/{subject_id}/{subject_id}_fnirs_windows.npy  → (N, 21, 16, 16, 30, 2)
  {OUTPUT_PATH}/{subject_id}/{subject_id}_labels.npy          → (N,)  کلاس 0/1/2
  {OUTPUT_PATH}/{subject_id}/{subject_id}_session_ids.npy     → (N,)  session 1/2/3

تنظیم کنید:
"""

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from itertools import product

# ─────────────────────────────────────────────
# ۱. تنظیمات — فقط این مسیر را عوض کنید
# ─────────────────────────────────────────────
OUTPUT_PATH = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED_RUN_A1b_10s"

CLASS_NAMES  = {0: "0-back", 1: "2-back", 2: "3-back"}
SESSION_NAMES = {1: "Session 1", 2: "Session 2", 3: "Session 3"}
ALPHA = 0.05  # سطح معناداری

# ─────────────────────────────────────────────
# ۲. پیدا کردن همه subjects
# ─────────────────────────────────────────────
def find_subjects(base_path):
    """پیدا کردن subjects بر اساس پوشه‌هایی که هر سه فایل npy را دارند."""
    subjects = []
    for folder in sorted(glob.glob(os.path.join(base_path, "VP*"))):
        subject_id = os.path.basename(folder)
        needed = [
            os.path.join(folder, f"{subject_id}_fnirs_windows.npy"),
            os.path.join(folder, f"{subject_id}_labels.npy"),
            os.path.join(folder, f"{subject_id}_session_ids.npy"),
        ]
        if all(os.path.exists(f) for f in needed):
            subjects.append(subject_id)
        else:
            print(f"[SKIP] {subject_id}: فایل ناقص")
    return subjects

# ─────────────────────────────────────────────
# ۳. بارگذاری و محاسبه آمار per window
# ─────────────────────────────────────────────
def load_subject(base_path, subject_id):
    """
    خروجی:
      hbo_per_window : (N,)  ← میانگین HbO روی تمام dim‌های فضایی/زمانی
      hbr_per_window : (N,)  ← میانگین HbR
      labels         : (N,)
      session_ids    : (N,)
    """
    folder = os.path.join(base_path, subject_id)

    fnirs = np.load(os.path.join(folder, f"{subject_id}_fnirs_windows.npy"),
                    mmap_mode='r')  # mmap برای حافظه — فایل‌ها بزرگ هستند
    labels = np.load(os.path.join(folder, f"{subject_id}_labels.npy"))
    sessions = np.load(os.path.join(folder, f"{subject_id}_session_ids.npy"))

    # fnirs shape: (N, 21, 16, 16, 30, 2)
    # میانگین روی تمام ابعاد غیر از N و channel (آخرین dim)
    # نتیجه: (N, 2) → [:, 0]=HbO   [:, 1]=HbR
    hbo_per_window = fnirs[..., 0].mean(axis=(1, 2, 3, 4)).astype(np.float64)
    hbr_per_window = fnirs[..., 1].mean(axis=(1, 2, 3, 4)).astype(np.float64)

    return hbo_per_window, hbr_per_window, labels.astype(int), sessions.astype(int)


def compute_stats(values, labels, sessions):
    """
    خروجی: dict با کلیدهای (session, class) → {'mean': x, 'var': x, 'n': x}
    """
    result = {}
    for sess in sorted(np.unique(sessions)):
        for cls in sorted(np.unique(labels)):
            mask = (sessions == sess) & (labels == cls)
            v = values[mask]
            if len(v) == 0:
                result[(sess, cls)] = {'mean': np.nan, 'var': np.nan, 'n': 0}
            else:
                result[(sess, cls)] = {
                    'mean': float(v.mean()),
                    'var':  float(v.var(ddof=1)),
                    'n':    int(len(v))
                }
    return result

# ─────────────────────────────────────────────
# ۴. آزمون آماری: session 3 vs sessions 1+2
# ─────────────────────────────────────────────
def session_shift_test(values, labels, sessions, cls):
    """
    Welch t-test + Cohen's d بین:
      group A = windows از sessions 1 و 2 با کلاس cls
      group B = windows از session 3 با کلاس cls
    خروجی: dict
    """
    mask_12 = ((sessions == 1) | (sessions == 2)) & (labels == cls)
    mask_3  = (sessions == 3) & (labels == cls)

    a = values[mask_12]
    b = values[mask_3]

    if len(a) < 2 or len(b) < 2:
        return {
            't': np.nan, 'p': np.nan, 'cohens_d': np.nan,
            'significant': False, 'n_12': len(a), 'n_3': len(b)
        }

    t_stat, p_val = stats.ttest_ind(a, b, equal_var=False)

    # Cohen's d (pooled std)
    pooled_std = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
    cohens_d = (b.mean() - a.mean()) / (pooled_std + 1e-12)

    return {
        't': float(t_stat),
        'p': float(p_val),
        'cohens_d': float(cohens_d),
        'significant': bool(p_val < ALPHA),
        'mean_12': float(a.mean()),
        'mean_3':  float(b.mean()),
        'n_12': len(a),
        'n_3':  len(b),
    }

# ─────────────────────────────────────────────
# ۵. رسم نمودار per subject
# ─────────────────────────────────────────────
def plot_subject(subject_id, hbo, hbr, labels, sessions, save_dir):
    """
    برای یک subject:
    - ردیف اول: HbO mean per session per class
    - ردیف دوم: HbR mean per session per class
    - ردیف سوم: box plot session 3 vs 1+2
    """
    fig, axes = plt.subplots(3, 3, figsize=(15, 11))
    fig.suptitle(f"{subject_id} — fNIRS Distribution Shift Analysis", fontsize=13, fontweight='bold')

    classes = sorted(np.unique(labels))
    session_colors = {1: '#2196F3', 2: '#4CAF50', 3: '#F44336'}

    for col, cls in enumerate(classes):
        cls_name = CLASS_NAMES[cls]

        # ── ردیف ۱: HbO mean + variance ────────────────────
        ax = axes[0, col]
        for sess in [1, 2, 3]:
            mask = (sessions == sess) & (labels == cls)
            v = hbo[mask]
            if len(v) > 0:
                ax.errorbar(sess, v.mean(), yerr=np.sqrt(v.var(ddof=1)),
                            fmt='o', color=session_colors[sess],
                            capsize=5, capthick=2, markersize=8,
                            label=f"S{sess} (n={len(v)})")
        ax.set_title(f"HbO — {cls_name}", fontsize=10)
        ax.set_xlabel("Session")
        ax.set_ylabel("Mean HbO ± SD")
        ax.set_xticks([1, 2, 3])
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

        # ── ردیف ۲: HbR mean + variance ────────────────────
        ax = axes[1, col]
        for sess in [1, 2, 3]:
            mask = (sessions == sess) & (labels == cls)
            v = hbr[mask]
            if len(v) > 0:
                ax.errorbar(sess, v.mean(), yerr=np.sqrt(v.var(ddof=1)),
                            fmt='s', color=session_colors[sess],
                            capsize=5, capthick=2, markersize=8,
                            label=f"S{sess} (n={len(v)})")
        ax.set_title(f"HbR — {cls_name}", fontsize=10)
        ax.set_xlabel("Session")
        ax.set_ylabel("Mean HbR ± SD")
        ax.set_xticks([1, 2, 3])
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

        # ── ردیف ۳: Box plot (session 1+2 vs session 3) ────
        ax = axes[2, col]
        mask_12 = ((sessions == 1) | (sessions == 2)) & (labels == cls)
        mask_3  = (sessions == 3) & (labels == cls)

        data_to_plot = [hbo[mask_12], hbo[mask_3], hbr[mask_12], hbr[mask_3]]
        positions = [1, 2, 3.5, 4.5]
        colors_bp = ['#90CAF9', '#EF9A9A', '#A5D6A7', '#FFAB91']
        labels_bp = ['HbO S1+2', 'HbO S3', 'HbR S1+2', 'HbR S3']

        bp = ax.boxplot(data_to_plot, positions=positions, patch_artist=True,
                        widths=0.6, showfliers=False)
        for patch, color in zip(bp['boxes'], colors_bp):
            patch.set_facecolor(color)

        # نتیجه t-test روی نمودار
        res_hbo = session_shift_test(hbo, labels, sessions, cls)
        res_hbr = session_shift_test(hbr, labels, sessions, cls)
        sig_hbo = "✓ SIG" if res_hbo['significant'] else "✗ n.s."
        sig_hbr = "✓ SIG" if res_hbr['significant'] else "✗ n.s."
        ax.set_title(
            f"{cls_name}\n"
            f"HbO: p={res_hbo['p']:.3f} {sig_hbo} | d={res_hbo['cohens_d']:.2f}\n"
            f"HbR: p={res_hbr['p']:.3f} {sig_hbr} | d={res_hbr['cohens_d']:.2f}",
            fontsize=8
        )
        ax.set_xticks(positions)
        ax.set_xticklabels(labels_bp, fontsize=7, rotation=15)
        ax.set_ylabel("Mean signal per window")
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    save_path = os.path.join(save_dir, f"{subject_id}_distribution_shift.png")
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  → نمودار ذخیره شد: {save_path}")


# ─────────────────────────────────────────────
# ۶. نمودار خلاصه همه subjects
# ─────────────────────────────────────────────
def plot_group_summary(all_results_df, save_dir):
    """
    Cohen's d همه subjects را برای هر کلاس × channel نشان می‌دهد.
    مقدار d > 0.2 small، > 0.5 medium، > 0.8 large effect.
    """
    channels = ['HbO', 'HbR']
    classes  = [0, 1, 2]

    fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharey=True)
    fig.suptitle("Group Summary — Cohen's d (Session 3 vs Sessions 1+2)\n"
                 "مقادیر مثبت = Session 3 بالاتر | منفی = Session 3 پایین‌تر",
                 fontsize=12, fontweight='bold')

    for row, ch in enumerate(channels):
        for col, cls in enumerate(classes):
            ax = axes[row, col]
            sub_df = all_results_df[
                (all_results_df['channel'] == ch) &
                (all_results_df['class'] == cls)
            ].copy()

            colors = ['#F44336' if sig else '#90A4AE'
                      for sig in sub_df['significant']]

            bars = ax.barh(sub_df['subject'], sub_df['cohens_d'],
                           color=colors, edgecolor='white', height=0.6)

            ax.axvline(0, color='black', linewidth=1)
            ax.axvline(0.5, color='orange', linewidth=1, linestyle='--', alpha=0.5)
            ax.axvline(-0.5, color='orange', linewidth=1, linestyle='--', alpha=0.5)
            ax.axvline(0.8, color='red', linewidth=1, linestyle=':', alpha=0.4)
            ax.axvline(-0.8, color='red', linewidth=1, linestyle=':', alpha=0.4)

            ax.set_title(f"{ch} — {CLASS_NAMES[cls]}", fontsize=10)
            ax.set_xlabel("Cohen's d")
            ax.grid(True, axis='x', alpha=0.3)

            # افزودن p-value کنار هر بار
            for i, (_, row_data) in enumerate(sub_df.iterrows()):
                p_str = f"p={row_data['p']:.3f}"
                ax.text(row_data['cohens_d'] + 0.02, i,
                        p_str, va='center', fontsize=7)

    # legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#F44336', label=f'Significant (p < {ALPHA})'),
        Patch(facecolor='#90A4AE', label='Not significant'),
    ]
    fig.legend(handles=legend_elements, loc='lower right', fontsize=9)

    plt.tight_layout()
    save_path = os.path.join(save_dir, "GROUP_distribution_shift_summary.png")
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"\n→ نمودار گروهی ذخیره شد: {save_path}")


# ─────────────────────────────────────────────
# ۷. ذخیره جدول CSV
# ─────────────────────────────────────────────
def save_table(all_results_df, stats_df, save_dir):
    csv_path = os.path.join(save_dir, "distribution_shift_results.csv")
    all_results_df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    print(f"→ جدول کامل ذخیره شد: {csv_path}")

    stats_path = os.path.join(save_dir, "per_subject_descriptive_stats.csv")
    stats_df.to_csv(stats_path, index=False, encoding='utf-8-sig')
    print(f"→ آمار توصیفی ذخیره شد: {stats_path}")


# ─────────────────────────────────────────────
# ۸. Main
# ─────────────────────────────────────────────
def main():
    print("=" * 65)
    print("fNIRS Distribution Shift Analysis — Session 3 vs Sessions 1+2")
    print("=" * 65)

    subjects = find_subjects(OUTPUT_PATH)
    if not subjects:
        print(f"\n[ERROR] هیچ subjectی در {OUTPUT_PATH} پیدا نشد.")
        print("مطمئن شوید OUTPUT_PATH درست تنظیم شده و فایل‌های .npy وجود دارند.")
        return

    print(f"\nSubjects پیدا شده: {subjects}\n")

    save_dir = os.path.join(OUTPUT_PATH, "_shift_analysis")
    os.makedirs(save_dir, exist_ok=True)

    all_test_rows   = []  # برای DataFrame نتایج آزمون
    all_stats_rows  = []  # برای DataFrame آمار توصیفی

    for subject_id in subjects:
        print(f"\n{'─'*55}")
        print(f"Processing {subject_id} ...")

        try:
            hbo, hbr, labels, sessions = load_subject(OUTPUT_PATH, subject_id)
        except Exception as e:
            print(f"  [ERROR] خطا در بارگذاری {subject_id}: {e}")
            continue

        print(f"  Windows: {len(labels)} | "
              f"Sessions: {np.unique(sessions)} | "
              f"Classes: {np.unique(labels)}")

        # آمار توصیفی
        for ch_name, values in [('HbO', hbo), ('HbR', hbr)]:
            stats_dict = compute_stats(values, labels, sessions)
            for (sess, cls), info in stats_dict.items():
                all_stats_rows.append({
                    'subject': subject_id,
                    'channel': ch_name,
                    'session': sess,
                    'class':   cls,
                    'class_name': CLASS_NAMES[cls],
                    'n': info['n'],
                    'mean': info['mean'],
                    'var':  info['var'],
                    'std':  np.sqrt(info['var']) if not np.isnan(info['var']) else np.nan,
                })

        # آزمون آماری
        for ch_name, values in [('HbO', hbo), ('HbR', hbr)]:
            for cls in sorted(np.unique(labels)):
                res = session_shift_test(values, labels, sessions, cls)
                row = {
                    'subject':    subject_id,
                    'channel':    ch_name,
                    'class':      cls,
                    'class_name': CLASS_NAMES[cls],
                    **res
                }
                all_test_rows.append(row)

                sig_str = "*** SIGNIFICANT ***" if res['significant'] else "not significant"
                print(f"  {ch_name} {CLASS_NAMES[cls]:7s}: "
                      f"t={res.get('t', np.nan):+.3f}  "
                      f"p={res.get('p', np.nan):.4f}  "
                      f"d={res.get('cohens_d', np.nan):+.3f}  "
                      f"→ {sig_str}")

        # نمودار per subject
        plot_subject(subject_id, hbo, hbr, labels, sessions, save_dir)

    if not all_test_rows:
        print("\n[ERROR] هیچ نتیجه‌ای تولید نشد.")
        return

    all_results_df = pd.DataFrame(all_test_rows)
    stats_df       = pd.DataFrame(all_stats_rows)

    # ── نمودار گروهی ──
    plot_group_summary(all_results_df, save_dir)

    # ── ذخیره جداول ──
    save_table(all_results_df, stats_df, save_dir)

    # ── خلاصه نهایی ──
    print("\n" + "=" * 65)
    print("نتیجه نهایی — تعداد موارد معنادار")
    print("=" * 65)
    sig_summary = (
        all_results_df.groupby(['channel', 'class_name'])['significant']
        .agg(['sum', 'count'])
        .rename(columns={'sum': 'significant', 'count': 'total'})
    )
    sig_summary['percent'] = 100 * sig_summary['significant'] / sig_summary['total']
    print(sig_summary.to_string())

    n_sig   = all_results_df['significant'].sum()
    n_total = len(all_results_df)
    print(f"\nمجموع: {n_sig}/{n_total} مورد معنادار "
          f"({100*n_sig/n_total:.1f}%)")

    if n_sig / n_total >= 0.5:
        print("\n>>> CONCLUSION: Distribution shift در Session 3 اثبات شد.")
        print("    این باید در thesis به عنوان cross-session domain shift مستند شود.")
    else:
        print("\n>>> CONCLUSION: شواهد کافی برای distribution shift یافت نشد.")
        print("    ممکن است shift فقط در برخی subjects وجود داشته باشد (جداول را بررسی کنید).")

    print(f"\nتمام نتایج در پوشه زیر ذخیره شدند:\n  {save_dir}")


if __name__ == "__main__":
    main()


fNIRS Distribution Shift Analysis — Session 3 vs Sessions 1+2

Subjects پیدا شده: ['VP001', 'VP002', 'VP003', 'VP004', 'VP005', 'VP006', 'VP007', 'VP008', 'VP009', 'VP010', 'VP011', 'VP012', 'VP013', 'VP014', 'VP015', 'VP016', 'VP017', 'VP018', 'VP019', 'VP020', 'VP021', 'VP022', 'VP023', 'VP024', 'VP025', 'VP026']


───────────────────────────────────────────────────────
Processing VP001 ...
  Windows: 827 | Sessions: [1 2 3] | Classes: [0 1 2]
  HbO 0-back : t=-1.954  p=0.0527  d=+0.264  → not significant
  HbO 2-back : t=-3.133  p=0.0020  d=+0.382  → *** SIGNIFICANT ***
  HbO 3-back : t=-3.378  p=0.0010  d=+0.455  → *** SIGNIFICANT ***
  HbR 0-back : t=-11.830  p=0.0000  d=+1.511  → *** SIGNIFICANT ***
  HbR 2-back : t=+4.387  p=0.0000  d=-0.489  → *** SIGNIFICANT ***
  HbR 3-back : t=+0.662  p=0.5086  d=-0.080  → not significant
  → نمودار ذخیره شد: E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED_RUN_A1b_10s\_shift_analysis\VP001_distribution_shift.png

───────────────────────